# ClearWater-Modules Demo 2: Coupling Water Quality Reactions to Transport with ClearWater-Riverine

**Objective**: Demonstrate a more complex scenario of coupled transport and reaction models in Sumwere Creek, using the [ClearWater-modules](https://github.com/EcohydrologyTeam/ClearWater-modules) to simulate heat exchange with the atmosphere.

This notebook builds on the introduction to using [ClearWater-riverine](https://github.com/EcohydrologyTeam/ClearWater-riverine) provided in the demo notebook in that reposittory.

## Background 
This notebook couples Clearwater-riverine (transport) with Clearwater-modules (reactions) - specifically, the Temperature Simulation Model (TSM). The Temperature Simulation Module (TSM) is an essential component of ClearWater (Corps Library for Environmental Analysis and Restoration of Watersheds). TSM plays a crucial role in simulating and predicting water temperature within aquatic ecosystems. TSM utilizes a comprehensive energy balance approach to account for various factors contributing to heat inputs and outputs in the water environment. It considers both external forcing functions and heat exchanges occurring at the water surface and the sediment-water interface. The primary contributors to heat exchange at the water surface include shortwave solar radiation, longwave atmospheric radiation, heat conduction from the atmosphere to the water, and direct heat inputs. Conversely, the primary factors that remove heat from the system are longwave radiation emitted by the water, evaporation, and heat conduction from the water to the atmosphere. 
The core principle behind TSM is the application of the laws of conservation of energy to compute water temperature. This means that the change in heat content of the water is directly related to changes in temperature, which, in turn, are influenced by various heat flux components. The specific heat of water is employed to establish this relationship. Each term of the heat flux equation can be calculated based on the input provided by the user, allowing for flexibility in modeling different environmental conditions

## Example Case Study

This example shows how to run Clearwater Riverine coupled with Clearwater Modules in a fictional location, "Sumwere Creek" (shown below). The flow field for Sumwere Creek comes from a HEC-RAS 2D model, which has a domain of 2x2 km and a base mesh cell size of 100x100 meters. 

![image.png](../docs/imgs/SumwereCreek_coarse.png)

The upstream boundary for Sumwere Creek is at the top left of the model domain, flowing into the domain at a constant 3 cms. At the first bend in the creek, there is an additional boundary representing a spring-fed tributary to the creek (1 cms). Further downstream, there is a meander in the stream forming a slow-flowing oxbow lake. There is another boundary flowing into that oxbow lake, representing a powerplant discharge (0.5 cms). 

The downstream boundary is a constant stage set at 20.75. The upstream inflows have a water temperature of 15 degrees C; the spring-fed creek has constant inflows of 5 C, and the powerplant is steady at 20 C with periodic higher temperature (25 C) discharges in a downstream meander.  

We simulate this scenario over the course of two full days, using meteorological parameters from Arizona (extreme temperature swings between night and day) to help show off the impacts of TSM.

## Data Availability
All data required run this notebook is available at this [Google Drive](https://drive.google.com/drive/folders/19uCjAJPZh4g6r1ZWzk1D_B8jZGluSc4N?usp=drive_link). 
This notebook will use the Clearwater-Modules `sumwere_creek_coarse_p48` model. Please download that entire only this folder from the Google Drive and place it in the `data_temp` folder (`examples/data_temp`) of this repository to run the rest of the notebook.

## Model Set-Up
### General Imports

add something here about the pixi environment setup...

In [1]:
from pathlib import Path

import pandas as pd
import xarray as xr

import hvplot.xarray
import hvplot.pandas
import holoviews as hv
from bokeh.models import HoverTool

from clearwater_modules_v2.config import init_from_file
from clearwater_riverine.plotting import RiverinePlotter


### Instantiate Models
#### Clearwater-Modules & Clearwater-Riverine

Ensure that you have followed the instructions in the Data Availability Section above, and that you have the files downloaded from the [Google Drive](https://drive.google.com/drive/folders/19uCjAJPZh4g6r1ZWzk1D_B8jZGluSc4N?usp=drive_link) for Clearwater-Modules `sumwere_creek_coarse_p48` and saved/unzipped to your local directory `examples/data_temp`.

This example sets up the model using a config file.

In addition, this example also demostrates that version 2 of Clearwater-Modules and the significant updates to Clearwater-Riverine reproduce the same results as the previous versions of these models. In order to fully replicate the previous model output it was necessary to reproduce the time-step level interpolated boundary condition time series that the previous version of Clearwater-Riverine generated from the user specified input boundary conditions. These time-step level boundary condition time series were created with this [notebook](https://github.com/EcohydrologyTeam/ClearWater-riverine/blob/main/examples/archive/02_coupling_riverine_modules_tsm_CompareTo_v2.ipynb) from the Clearwater-Riverine repository. This [notebook](https://github.com/EcohydrologyTeam/ClearWater-riverine/blob/main/examples/archive/02_coupling_riverine_modules_tsm_CompareTo_v2.ipynb) has been previously executed and the time-step level boundary condition time series were saved in the Google Drive copy of the version 2 `sumwere_creek_coarse_p48` inputs for convience. This [notebook](https://github.com/EcohydrologyTeam/ClearWater-riverine/blob/main/examples/archive/02_coupling_riverine_modules_tsm_CompareTo_v2.ipynb) can be executed again to verify the inputs that were saved--please follow the instructions in that notebook.

This [notebook](https://github.com/EcohydrologyTeam/ClearWater-riverine/blob/main/examples/archive/02_coupling_riverine_modules_tsm_CompareTo_v2.ipynb) also generated the previous coupled models version water temperature output that is used for comparison later in this notebook. The previous model output was also saved in the Google drive for convience.

### Run the Coupled Models

In [2]:
#### Set local filepath to the configuration yaml file ####
#  Set the following "use_v1_intrp" variable to "False" to explore the effects of the improved 
#  interpolation scheme for boundary condition input time series. Keeping it as "True" will
#  demostrates version 2 can produce the same results as version 1:
use_v1_intrp = True
#use_v1_intrp = False


# Simulation directory path for inputs and outputs
model_name = 'sumwere_creek_coarse_p48'
sim_path = Path.cwd().parent / 'data_temp' / model_name


if use_v1_intrp == True:
    config_path = sim_path / 'modules_v1_interp.yml'
else:
    config_path = sim_path / 'modules.yml'
    
print(config_path.exists())

#### Initialize a version 2 model of the linked Riverine and TSM models ####
model = init_from_file(config_path)


True


In [3]:
#### Simulate a version 2 model of the linked Riverine and TSM models ####
model.run()

[2026-04-14 11:50:55] INFO - Running timestep: 2022-05-13 08:00:00
[2026-04-14 11:50:55] INFO - Running timestep: 2022-05-13 08:00:30
[2026-04-14 11:50:55] INFO - Running timestep: 2022-05-13 08:01:00
[2026-04-14 11:50:55] INFO - Running timestep: 2022-05-13 08:01:30
[2026-04-14 11:50:55] INFO - Running timestep: 2022-05-13 08:02:00
[2026-04-14 11:50:55] INFO - Running timestep: 2022-05-13 08:02:30
[2026-04-14 11:50:55] INFO - Running timestep: 2022-05-13 08:03:00
[2026-04-14 11:50:55] INFO - Running timestep: 2022-05-13 08:03:30
[2026-04-14 11:50:55] INFO - Running timestep: 2022-05-13 08:04:00
[2026-04-14 11:50:55] INFO - Running timestep: 2022-05-13 08:04:30
[2026-04-14 11:50:56] INFO - Running timestep: 2022-05-13 08:05:00
[2026-04-14 11:50:56] INFO - Running timestep: 2022-05-13 08:05:30
[2026-04-14 11:50:56] INFO - Running timestep: 2022-05-13 08:06:00
[2026-04-14 11:50:56] INFO - Running timestep: 2022-05-13 08:06:30
[2026-04-14 11:50:56] INFO - Running timestep: 2022-05-13 08:0

### Plot the Coupled Models Results

In [4]:
#Initialize a Riverine dynamic plotting tool
plotter = RiverinePlotter(registry=model._Model__registry, crs='EPSG:26916')

In [5]:
#Plot the water temperature results from the linked Riverine and TSM model simulation
plotter.dynamic_plot(constituent_name = 'water_temperature')

:DynamicMap   [datetime]
   :Overlay
      .Polygons.I :Polygons   [Longitude,Latitude]   (water_temperature,nface)
      .WMTS.I     :WMTS   [Longitude,Latitude]

### Compare the v2 Model Results with the Previous Version

In [6]:
#### Open a Zarr directory for a previously saved version 1 reaction model output ####

reaction_model_v1_Path = sim_path / 'model_output_v1_asPrevPresented/model_output_reaction.zarr'
print(reaction_model_v1_Path.exists())

reaction_model_v1 = xr.open_zarr(reaction_model_v1_Path)
reaction_model_v1

True


<xarray.Dataset> Size: 14MB
Dimensions:            (nface: 367, seconds: 962)
Coordinates:
    face_x             (nface) float64 3kB dask.array<chunksize=(367,), meta=np.ndarray>
    face_y             (nface) float64 3kB dask.array<chunksize=(367,), meta=np.ndarray>
  * seconds            (seconds) int64 8kB 0 1 2 3 4 5 ... 957 958 959 960 961
    time               datetime64[ns] 8B ...
Dimensions without coordinates: nface
Data variables: (12/32)
    a0                 (nface) float64 3kB dask.array<chunksize=(367,), meta=np.ndarray>
    a1                 (nface) float64 3kB dask.array<chunksize=(367,), meta=np.ndarray>
    a2                 (nface) float64 3kB dask.array<chunksize=(367,), meta=np.ndarray>
    a3                 (nface) float64 3kB dask.array<chunksize=(367,), meta=np.ndarray>
    a4                 (nface) float64 3kB dask.array<chunksize=(367,), meta=np.ndarray>
    a5                 (nface) float64 3kB dask.array<chunksize=(367,), meta=np.ndarray>
    ...                 ...
    water_temp_c       (seconds, nface) float64 3MB dask.array<chunksize=(241, 184), meta=np.ndarray>
    wind_a             (nface) float64 3kB dask.array<chunksize=(367,), meta=np.ndarray>
    wind_b             (nface) float64 3kB dask.array<chunksize=(367,), meta=np.ndarray>
    wind_c             (nface) float64 3kB dask.array<chunksize=(367,), meta=np.ndarray>
    wind_kh_kw         (nface) float64 3kB dask.array<chunksize=(367,), meta=np.ndarray>
    wind_speed         (nface) float64 3kB dask.array<chunksize=(367,), meta=np.ndarray>

In [7]:
def compare_v1_v2_for_nface_i(
        nface_i: int,
        reaction_model_v1: xr.Dataset,
        model: clearwater_modules_v2.model.Model
):
    """
    Returns a dataframe for a nface with water temperature model results from a version 1 and version 2 simulations
    and plots
    """
    key_v1 = 'water_temp_c'
    key_v2 = 'water_temperature'
    
    #Format Select nface model results into a dataframe for version 1
    waterTemp_v1 = reaction_model_v1[key_v1].isel(nface=nface_i).to_dataframe()
    waterTemp_v1['old_time'] = waterTemp_v1.index
    waterTemp_v1['total_seconds'] = waterTemp_v1.index * 30  #convert from timestep index to total seconds (30 second timesteps)
    waterTemp_v1['time'] = waterTemp_v1['time'] + pd.to_timedelta(waterTemp_v1.total_seconds, unit='s')
    waterTemp_v1['temp_v1'] = waterTemp_v1['water_temp_c']
    waterTemp_v1_B = waterTemp_v1[['time', 'temp_v1']]
    waterTemp_v1_C = waterTemp_v1_B.iloc[:-1] # remove last record
    
    #Format Select nface model results into a dataframe for version 2
    waterTemp_v2 = model._Model__registry._registry[key_v2].get().isel(nface=nface_i).to_dataframe()
    waterTemp_v2_B = waterTemp_v2.rename(columns={'water_temperature': 'temp_v2'})
    waterTemp_v2_C = waterTemp_v2_B[['temp_v2']]
    
    #Merge dataframes
    df_merged = pd.merge(waterTemp_v1_C, waterTemp_v2_C, on='time', suffixes=('_v1', '_v2'))
    
    #Calculate difference between v1 and v2 water temperature
    df_merged['diff_v1-v2'] = df_merged['temp_v1'] - df_merged['temp_v2']


    #### PLOTTING ####
    hover = HoverTool(tooltips=[
        ("Date/Time", "@time{%F %T}"),
        ("v1", "@temp_v1{0.00000000}"),
        ("v2", "@temp_v2{0.00000000}"),
    ],
        formatters={
            "@time": "datetime",
        },
        mode='vline'
    )
    
    curve1 = hv.Curve(
        df_merged,
        kdims=['time'],
        vdims=['temp_v1', 'temp_v2'],
        label='v1'
    )
    
    curve2 = hv.Curve(
        df_merged,
        kdims=['time'],
        vdims=['temp_v2', 'temp_v1'],
        label='v2'
    ).opts(line_dash='dashed')
    
    plot_curves = (curve1 * curve2).opts(
        hv.opts.Overlay(
            width=700,
            height=500,
            xlabel='Date/Time',
            ylabel='Water Temperature',        
            legend_position='bottom',
            show_legend=True,
            legend_opts={'location': 'bottom_center', 'orientation': 'horizontal'}
        ),
        hv.opts.Curve(tools=[hover])
    )

    plot_diff = df_merged.hvplot(x='time', y='diff_v1-v2', label='diff_v1-v2', hover=True)
    plot_compare = (plot_curves + plot_diff).cols(1)
    
    return df_merged, plot_compare
    

In [8]:
nface_i = 275 #### A Model Cell in the Upstream End of Sumwere Creek ####

df, plot = compare_v1_v2_for_nface_i(nface_i, reaction_model_v1, model)
plot

:Layout
   .Overlay.I                     :Overlay
      .Curve.V1 :Curve   [time]   (temp_v1,temp_v2)
      .Curve.V2 :Curve   [time]   (temp_v2,temp_v1)
   .Curve.Diff_v1_hyphen_minus_v2 :Curve   [time]   (diff_v1-v2)

In [9]:
nface_i = 226 #### A Model Cell at the Confluence of Sumwere Creek and Cold Spring ####

df, plot = compare_v1_v2_for_nface_i(nface_i, reaction_model_v1, model)
plot

:Layout
   .Overlay.I                     :Overlay
      .Curve.V1 :Curve   [time]   (temp_v1,temp_v2)
      .Curve.V2 :Curve   [time]   (temp_v2,temp_v1)
   .Curve.Diff_v1_hyphen_minus_v2 :Curve   [time]   (diff_v1-v2)

In [10]:
nface_i = 150 #### A Model Cell in the Oxbow Lake near the Power Plant discharge ####

df, plot = compare_v1_v2_for_nface_i(nface_i, reaction_model_v1, model)
plot

:Layout
   .Overlay.I                     :Overlay
      .Curve.V1 :Curve   [time]   (temp_v1,temp_v2)
      .Curve.V2 :Curve   [time]   (temp_v2,temp_v1)
   .Curve.Diff_v1_hyphen_minus_v2 :Curve   [time]   (diff_v1-v2)

In [11]:
nface_i = 264 #### A Model Cell in the Downstream End of Sumwere Creek ####

df, plot = compare_v1_v2_for_nface_i(nface_i, reaction_model_v1, model)
plot

:Layout
   .Overlay.I                     :Overlay
      .Curve.V1 :Curve   [time]   (temp_v1,temp_v2)
      .Curve.V2 :Curve   [time]   (temp_v2,temp_v1)
   .Curve.Diff_v1_hyphen_minus_v2 :Curve   [time]   (diff_v1-v2)